# Exploring different features and model to select the best

Generating the report for my data

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , MinMaxScaler , FunctionTransformer , StandardScaler , OrdinalEncoder
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split , cross_val_score , RandomizedSearchCV
from sklearn.metrics         import mean_squared_error, r2_score, accuracy_score
from scipy.stats import uniform
from sklearn.tree            import DecisionTreeClassifier, plot_tree
from sklearn.ensemble        import RandomForestClassifier


In [3]:
titanic_df = pd.read_csv("F://Titanic_Project//Data//train (2).csv")
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
import sweetviz as sv

report = sv.analyze(titanic_df)
report.show_html("F://Titanic_Project//Reports//raw_report.html" , open_browser=False)

Using the report and studying the data following things are identified

- Age is skewed right have to use np.log function to fix it
- Age has 20% missing values have to fix it
- Age has fractions for childerens under 1 
- sibling spouse and parent are in the sepearate coloumn can join them to make the family coloumn
- fare is also highly skewed have to fix it
- Cabin has large missing value so can create a falg for cabin present or not present
- embarked has two missing values can be filled with the mode

In [22]:
x_temp = titanic_df.copy()
x_temp["Family"] = x_temp["SibSp"] + x_temp["Parch"] + 1
x_temp["Is_married"] = x_temp["Name"].str.contains("Mrs", na=False).astype(int)
x_temp["Family_size"] = np.where(
    x_temp["Family"] >= 4,
    "Large",
    "Small"
)
x_temp["Is_alone"] = np.where(
    x_temp["Family"] == 1,
    1,
    0
)
x_temp["Age_bracket"] = pd.cut(x_temp['Age'], bins=[-np.inf, 12, 18, 60, np.inf], labels=['child','teen','adult','senior'])
x_temp["Has_cabin"] = x_temp["Cabin"].notna().astype(int)
x_temp["Fare_bracket"] = pd.qcut(x_temp["Fare"] ,q=4 , labels=["Low_fare" , "Medium_fare" , "High_fare" , "Very_high_fare"], duplicates='drop')
x_temp.drop(columns=["PassengerId" , "Name" , "Age" , "SibSp" , "Parch" , "Ticket" , "Fare" , "Cabin" , "Family"],inplace=True)
x_temp

,Survived,Pclass,Sex,Embarked,Is_married,Family_size,Is_alone,Age_bracket,Has_cabin,Fare_bracket
0,0,3,male,S,0,Small,0,adult,0,Low_fare
1,1,1,female,C,1,Small,0,adult,1,Very_high_fare
2,1,3,female,S,0,Small,1,adult,0,Medium_fare
3,1,1,female,S,1,Small,0,adult,1,Very_high_fare
4,0,3,male,S,0,Small,1,adult,0,Medium_fare
...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,S,0,Small,1,adult,0,Medium_fare
887,1,1,female,S,0,Small,1,adult,1,High_fare
888,0,3,female,S,0,Large,0,NaN,0,High_fare
889,1,1,male,C,0,Small,1,adult,1,High_fare


Following is the feature engineering that i have done based on the coloumns now i will add all this to the function that i can apply to the data 
- Since Embarked and Age need to fill the missing values before applying the function i will make a seprate pipeline to fill these values first

In [10]:
def transform_data(df):
    X = df.copy()
    X["Age"] = X["Age"].fillna(X["Age"].median())
    X["Embarked"] = X["Embarked"].fillna(X["Embarked"].mode()[0])
    X["Family"] = X["SibSp"] + X["Parch"] + 1
    X["Is_married"] = X["Name"].str.contains("Mrs", na=False).astype(int)
    X["Family_size"] = np.where(
        X["Family"] >= 4,
        "Large",
        "Small"
    )
    X["Is_alone"] = np.where(
        X["Family"] == 1,
        1,
        0
    )
    X["Age_bracket"] = pd.cut(X['Age'], bins=[-np.inf, 12, 18, 60, np.inf], labels=['child','teen','adult','senior'])
    X["Has_cabin"] = X["Cabin"].notna().astype(int)
    X["Fare_bracket"] = pd.qcut(X["Fare"] ,q=4 , labels=["Low_fare" , "Medium_fare" , "High_fare" , "Very_high_fare"], duplicates='drop')
    X.drop(columns=["PassengerId" , "Name" , "Age" , "SibSp" , "Parch" , "Ticket" , "Fare" , "Cabin" , "Family"],inplace=True)    
    return X

Specifying all the different coloumns

In [4]:
numeric_coloumns = ["Pclass" , "Is_married" , "Is_alone" , "Has_cabin"]
categorical_coloumns_one_hot = ["Sex" , "Embarked"]
categorical_coloumns_ordinal = ["Family_size", "Age_bracket", "Fare_bracket"]

Now making the pipeline for each one of these

In [5]:
num_pipeline = Pipeline([
    ("Scaler" , StandardScaler())
])

In [6]:
categorical_one_hot_pipeline = Pipeline([
    ("Encoder" , OneHotEncoder(handle_unknown="ignore"))
])

In [7]:
categorical_ordinal_pipeline = Pipeline([
    ("Encoder" , OrdinalEncoder(
        categories=[
            ["Small" , "Large"],
            ['child','teen','adult','senior'],
            ["Low_fare" , "Medium_fare" , "High_fare" , "Very_high_fare"],
        ]
    ))
])

In [8]:
preprocessor = ColumnTransformer([
    ("num" , num_pipeline , numeric_coloumns),
    ("cat_one" , categorical_one_hot_pipeline , categorical_coloumns_one_hot),
    ("cat_ord" , categorical_ordinal_pipeline , categorical_coloumns_ordinal)
])

In [68]:
linear_pipeline = Pipeline([
    ("Feature_enginnering" , FunctionTransformer(transform_data)),
    ("Preprocessing" , preprocessor),
    ("Model" , LogisticRegression(C=9.51, max_iter=1000) )
])

In [13]:
X = titanic_df.drop(columns=["Survived"])
y = titanic_df["Survived"]

x_train , x_test , y_train , y_test = train_test_split(X , y , test_size=0.30 , random_state=42)


In [72]:
scores = np.mean(cross_val_score(linear_pipeline , x_train , y_train , cv=5))
scores

np.float64(0.8201935483870969)

Checking for different parameters

In [71]:
param_dist = {
    "Model__C": uniform(0.01, 10),                    
}

random_search = RandomizedSearchCV(
    linear_pipeline,
    param_distributions=param_dist,
    n_iter=20,             
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1              
)

random_search.fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)

{'Model__C': np.float64(9.51714306409916)}
0.8201935483870969


In [75]:
linear_pipeline.fit(x_train, y_train)
final_score = linear_pipeline.score(x_test, y_test)
final_score

0.7947761194029851

Using Trees

In [29]:
Tree_pipline = Pipeline([
    ("Feature_enginnering" , FunctionTransformer(transform_data)),
    ("Preprocessing" , preprocessor),
    ("Model" , DecisionTreeClassifier(max_depth=3 , criterion="gini" , min_samples_split=15 , random_state=42 , min_samples_leaf=5) )
])


In [30]:
score_tree = np.mean(cross_val_score(Tree_pipline , x_train , y_train , cv=5))
score_tree

np.float64(0.8330709677419355)

Finding best parameters

In [34]:
param_dist = {
    "Model__max_depth": [3, 5, 10, 15, 20, 50],
    "Model__criterion": ["entropy", "gini", "log_loss"],
    "Model__min_samples_split": [2, 5, 10, 15, 20, 50],
    "Model__min_samples_leaf": [1, 2, 5, 10, 20]
}

random_tree_search = RandomizedSearchCV(
    Tree_pipline,
    param_distributions=param_dist,
    n_iter=20,             
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1              
)

random_tree_search.fit(x_train, y_train)
print(random_tree_search.best_params_)
print(random_tree_search.best_score_)

{'Model__min_samples_split': 15, 'Model__min_samples_leaf': 5, 'Model__max_depth': 3, 'Model__criterion': 'gini'}
0.8330709677419355


In [31]:
Tree_pipline.fit(x_train, y_train)
final_score = Tree_pipline.score(x_test, y_test)
final_score

0.8059701492537313

Now instead of using only one tree lets use the random forest multiple trees to check the score

In [37]:
random_forest_pipeline = Pipeline([
    ("Feature_enginnering" , FunctionTransformer(transform_data)),
    ("Preprocessing" , preprocessor),
    ("Model" , RandomForestClassifier(n_estimators=50 , min_samples_split=20 , min_samples_leaf=1,max_depth=10 , max_features="log2", random_state=42 , criterion="gini" ) )
])

In [38]:
score_random_tree = np.mean(cross_val_score(random_forest_pipeline , x_train , y_train , cv=5))
score_random_tree

np.float64(0.8346451612903225)

Finding the best parameters


In [36]:
param_dist = {
    "Model__n_estimators" : [50,100,200,300,400,500],
    "Model__max_depth": [3, 5, 10, 15, 20, 50 , None],
    "Model__max_features": ["sqrt" , "log2"],
    "Model__min_samples_split": [2, 5, 10, 15, 20, 50],
    "Model__min_samples_leaf": [1, 2, 5, 10, 20],
    "Model__criterion": ["entropy", "gini", "log_loss"],
}

random_forest_search = RandomizedSearchCV(
    random_forest_pipeline,
    param_distributions=param_dist,
    n_iter=20,             
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1              
)

random_forest_search.fit(x_train, y_train)
print(random_forest_search.best_params_)
print(random_forest_search.best_score_)

{'Model__n_estimators': 50, 'Model__min_samples_split': 20, 'Model__min_samples_leaf': 1, 'Model__max_features': 'log2', 'Model__max_depth': 10, 'Model__criterion': 'gini'}
0.8346451612903225


In [39]:
random_forest_pipeline.fit(x_train, y_train)
final_score = random_forest_pipeline.score(x_test, y_test)
final_score

0.8022388059701493